# NU Industries Production Planning Optimization

### Group Members

Samin Bahizad  
Bobby Hendricks  
Deysi Paniagua-Perez  
Nathan Stark  

## Project Summary

NU Industries manufactures Widgets, Gadgets, and Flugels at two plants over five production periods. The purpose of this project is to determine the production, inventory, advertising, labor, raw material, and shipping plan that maximizes total profit while meeting demand and staying within the company’s operating limits.

The baseline model includes regular and overtime labor, two raw materials, inventory held at each plant, advertising that increases demand in the following period, and shipments from each plant to the distribution center. After the baseline solution is reviewed, the group plans to complete sensitivity analysis, test changes to the advertising budget and raw material availability, and compare the linear programming solution with an integer programming model.

## Project Details

> **Hint:** This problem can be completed with fewer than 150 variables and fewer than 110 constraints.

## Problem Description

NU Industries operates two manufacturing plants that produce three products: Widgets, Gadgets, and Flugels. The finished products are shipped to the Distribution Center for final distribution to customers. Five production periods are to be scheduled.

According to the Sales Department, the production requirements that must be met due to contracts during the planning horizon are:

| Product | Period 1 | Period 2 | Period 3 | Period 4 | Period 5 |
|---|---:|---:|---:|---:|---:|
| Widgets | 70 | 125 | 185 | 190 | 200 |
| Gadgets | 200 | 300 | 295 | 245 | 240 |
| Flugels | 140 | 175 | 205 | 235 | 230 |

The Marketing and Forecasting Department anticipates that NU Industries can create additional demand through advertising.

- Each $160 invested in Widget advertising during one period creates demand for one additional Widget in the next period.
- Each $120 invested in Gadget advertising creates demand for one additional Gadget in the next period.
- Each $180 invested in Flugel advertising creates demand for one additional Flugel in the next period.
- The total advertising budget is limited to $70,000 for the entire planning horizon.

Throughout the planning horizon, NU Industries will sell:

| Product | Selling Price |
|---|---:|
| Widget | $2,490 |
| Gadget | $1,990 |
| Flugel | $2,970 |

The products can be manufactured at either Plant A or Plant B.

## Plant A

The production requirements at Plant A are:

- Each Widget requires 194 pounds of Raw Material 1, 8.6 pounds of Raw Material 2, and 9.5 hours of labor.
- Each Gadget requires 230 pounds of Raw Material 1 and 7.1 hours of labor.
- Each Flugel requires 178 pounds of Raw Material 1, 11.6 pounds of Raw Material 2, and 11.1 hours of labor.

Regular labor availability is limited to 2,500 hours in each period, but overtime can be scheduled in any amount if necessary.

Labor costs during periods 1 and 2 are:

- Regular labor: $11.00 per hour
- Overtime labor: $16.50 per hour

Labor costs increase by 5% after period 2.

The inventory area at Plant A can store a combined maximum of 70 units.

| Product | Inventory Cost per Unit |
|---|---:|
| Widget | $7.50 |
| Gadget | $5.50 |
| Flugel | $6.50 |

## Plant B

Plant B is the more modern facility and can produce the products slightly more efficiently.

The production requirements at Plant B are:

- Each Widget requires 188 pounds of Raw Material 1, 9.2 pounds of Raw Material 2, and 9.1 hours of labor.
- Each Gadget requires 225 pounds of Raw Material 1 and 7.8 hours of labor.
- Each Flugel requires 170 pounds of Raw Material 1, 10.8 pounds of Raw Material 2, and 10.6 hours of labor.

Regular labor availability is limited to 3,800 hours in each period, but overtime can be scheduled in any amount if necessary.

Labor costs during periods 1 and 2 are:

- Regular labor: $11.00 per hour
- Overtime labor: $16.50 per hour

Labor costs increase by 10% after period 2.

The inventory area at Plant B can store a combined maximum of 50 units.

| Product | Inventory Cost per Unit |
|---------|-----------------------:|
| Widget |         $7.80 |
| Gadget |         $5.70 |
| Flugel |         $7.00 |

## Raw Materials

A maximum of 70 tons of Raw Material 1 and 2.5 tons of Raw Material 2 are available from the vendor during each period.

Note: 1 ton = 2,000 pounds.

| Raw Material | Plant A Cost per Pound | Plant B Cost per Pound |
|---|---:|---:|
| Raw Material 1 | $1.25 | $1.45 |
| Raw Material 2 | $2.65 | $2.90 |

Each plant only purchases raw materials that can be used during the current period because storage space is limited.

## Transportation Costs

The average transportation cost for shipping one unit of finished product from each plant to the Distribution Center is:

| Product | Plant A | Plant B |
|---|---:|---:|
| Widget | $6.30 | $6.50 |
| Gadget | $4.60 | $5.00 |
| Flugel | $5.50 | $5.70 |

## Model Requirements

Demand during a given period must be satisfied using products manufactured during that period or inventory carried over from a previous period.

The goal is to determine the marketing, production, distribution, and inventory strategy that maximizes total profit.

The following assumptions apply:

- There is no inventory at the beginning of period 1.
- There should be no inventory at the end of period 5.
- All other plant overhead is constant and can be ignored.
- Fractional units are allowed in the baseline linear programming model.
- Results may be rounded to the nearest tenth for reporting purposes.

## Additional Analysis

After solving the baseline case, the group will complete a sensitivity analysis and make business recommendations.

The analysis should consider questions such as:

- Should the advertising budget be increased?
- If so, by how much?
- Which raw material is most limiting?
- How would additional raw material availability affect profit?
- How do the recommendations change under different scenarios?

The group will also solve the model as an integer programming problem and compare the results with the baseline linear programming solution.

In [1]:
%pip install pulp pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# import pulp and pandas
from pulp import LpVariable, LpProblem, LpMaximize, LpStatus, value, lpSum, GLPK
import pandas as pd


## Model variables

In [3]:
# define variable
products = ["Widget", "Gadget", "Flugel"]
plants = ["Plant_A", "Plant_B"]
periods = [1, 2, 3, 4, 5]
advertising_periods = [1, 2, 3, 4]

## Case Study Data

In [4]:

# required contract demand by product and period
base_demand = {
    ("Widget", 1): 70,
    ("Widget", 2): 125,
    ("Widget", 3): 185,
    ("Widget", 4): 190,
    ("Widget", 5): 200,

    ("Gadget", 1): 200,
    ("Gadget", 2): 300,
    ("Gadget", 3): 295,
    ("Gadget", 4): 245,
    ("Gadget", 5): 240,

    ("Flugel", 1): 140,
    ("Flugel", 2): 175,
    ("Flugel", 3): 205,
    ("Flugel", 4): 235,
    ("Flugel", 5): 230
}

# selling price per unit
selling_price = {
    "Widget": 2490,
    "Gadget": 1990,
    "Flugel": 2970
}

# advertising cost required to create one additional unit of demand
advertising_cost = {
    "Widget": 160,
    "Gadget": 120,
    "Flugel": 180
}

# raw material 1 requirements in pounds per unit
raw_material_1 = {
    ("Plant_A", "Widget"): 194,
    ("Plant_A", "Gadget"): 230,
    ("Plant_A", "Flugel"): 178,

    ("Plant_B", "Widget"): 188,
    ("Plant_B", "Gadget"): 225,
    ("Plant_B", "Flugel"): 170
}

# raw material 2 requirements in pounds per unit - Gadgets do not require raw material 2.
raw_material_2 = {
    ("Plant_A", "Widget"): 8.6,
    ("Plant_A", "Gadget"): 0,
    ("Plant_A", "Flugel"): 11.6,

    ("Plant_B", "Widget"): 9.2,
    ("Plant_B", "Gadget"): 0,
    ("Plant_B", "Flugel"): 10.8
}

# labor requirements in hours per unit
labor_hours = {
    ("Plant_A", "Widget"): 9.5,
    ("Plant_A", "Gadget"): 7.1,
    ("Plant_A", "Flugel"): 11.1,

    ("Plant_B", "Widget"): 9.1,
    ("Plant_B", "Gadget"): 7.8,
    ("Plant_B", "Flugel"): 10.6
}

# regular labor availability by plant and period
regular_labor_capacity = {
    ("Plant_A", period): 2500 for period in periods
}
regular_labor_capacity.update({
    ("Plant_B", period): 3800 for period in periods
})

# regular labor cost per hour
regular_labor_cost = {
    ("Plant_A", period): 11 if period <= 2 else 11 * 1.05
    for period in periods
}
regular_labor_cost.update({
    ("Plant_B", period): 11 if period <= 2 else 11 * 1.10
    for period in periods
})

# overtime labor cost per hour
overtime_labor_cost = {
    ("Plant_A", period): 16.50 if period <= 2 else 16.50 * 1.05
    for period in periods
}
overtime_labor_cost.update({
    ("Plant_B", period): 16.50 if period <= 2 else 16.50 * 1.10
    for period in periods
})

# inventory capacity by plant
inventory_capacity = {
    "Plant_A": 70,
    "Plant_B": 50
}

# inventory cost for holding one unit from one period to the next
inventory_cost = {
    ("Plant_A", "Widget"): 7.50,
    ("Plant_A", "Gadget"): 5.50,
    ("Plant_A", "Flugel"): 6.50,

    ("Plant_B", "Widget"): 7.80,
    ("Plant_B", "Gadget"): 5.70,
    ("Plant_B", "Flugel"): 7.00
}

# raw material costs by plant
raw_material_1_cost = {
    "Plant_A": 1.25,
    "Plant_B": 1.45
}

raw_material_2_cost = {
    "Plant_A": 2.65,
    "Plant_B": 2.90
}

# transportation cost per unit shipped from a plant to the distribution center
shipping_cost = {
    ("Plant_A", "Widget"): 6.30,
    ("Plant_A", "Gadget"): 4.60,
    ("Plant_A", "Flugel"): 5.50,

    ("Plant_B", "Widget"): 6.50,
    ("Plant_B", "Gadget"): 5.00,
    ("Plant_B", "Flugel"): 5.70
}

# total limits that apply during each period
raw_material_1_capacity = 70 * 2000     # 70 tons converted to pounds
raw_material_2_capacity = 2.5 * 2000    # 2.5 tons converted to pounds
total_advertising_budget = 70000


## Decision Variables

In [5]:
# define production variables
production = {
    (product, plant, period): LpVariable(
        f"Production_{product}_{plant}_P{period}", 0, None
    )
    for product in products
    for plant in plants
    for period in periods
}

# define shipment variables
shipment = {
    (product, plant, period): LpVariable(
        f"Shipment_{product}_{plant}_P{period}", 0, None
    )
    for product in products
    for plant in plants
    for period in periods
}

# define inventory variables
inventory = {
    (product, plant, period): LpVariable(
        f"Inventory_{product}_{plant}_P{period}", 0, None
    )
    for product in products
    for plant in plants
    for period in periods
}

# define regular labor variables
regular_labor = {
    (plant, period): LpVariable(
        f"Regular_Labor_{plant}_P{period}",
        0,
        regular_labor_capacity[(plant, period)]
    )
    for plant in plants
    for period in periods
}

# define overtime labor variables
overtime_labor = {
    (plant, period): LpVariable(
        f"Overtime_Labor_{plant}_P{period}", 0, None
    )
    for plant in plants
    for period in periods
}

# define advertising variables
advertising = {
    (product, period): LpVariable(
        f"Advertising_Demand_{product}_P{period}", 0, None
    )
    for product in products
    for period in advertising_periods
}

## Optimization Problem and Objective Function

In [6]:
# defines the problem
nu_model = LpProblem("NU_Industries_Baseline", LpMaximize)

# calculate total sales revenue
total_revenue = lpSum(
    selling_price[product] * shipment[(product, plant, period)]
    for product in products
    for plant in plants
    for period in periods
)

# calculate total raw material cost
total_raw_material_cost = lpSum(
    (
        raw_material_1[(plant, product)] * raw_material_1_cost[plant]
        + raw_material_2[(plant, product)] * raw_material_2_cost[plant]
    ) * production[(product, plant, period)]
    for product in products
    for plant in plants
    for period in periods
)

# calculate total regular labor cost
total_regular_labor_cost = lpSum(
    regular_labor_cost[(plant, period)] * regular_labor[(plant, period)]
    for plant in plants
    for period in periods
)

# calculate total overtime labor cost
total_overtime_labor_cost = lpSum(
    overtime_labor_cost[(plant, period)] * overtime_labor[(plant, period)]
    for plant in plants
    for period in periods
)

# calculate total inventory cost
total_inventory_cost = lpSum(
    inventory_cost[(plant, product)] * inventory[(product, plant, period)]
    for product in products
    for plant in plants
    for period in periods[:-1]
)

# calculate total transportation cost
total_shipping_cost = lpSum(
    shipping_cost[(plant, product)] * shipment[(product, plant, period)]
    for product in products
    for plant in plants
    for period in periods
)

# calculate total advertising cost
total_advertising_cost = lpSum(
    advertising_cost[product] * advertising[(product, period)]
    for product in products
    for period in advertising_periods
)

# define objective function
nu_model += (
    total_revenue
    - total_raw_material_cost
    - total_regular_labor_cost
    - total_overtime_labor_cost
    - total_inventory_cost
    - total_shipping_cost
    - total_advertising_cost
)

## Constraints

In [7]:
# inventory balance constraints
# beginning inventory + production = shipment + ending inventory
for product in products:
    for plant in plants:
        for period in periods:
            if period == 1:
                nu_model += (
                    production[(product, plant, period)]
                    == shipment[(product, plant, period)]
                    + inventory[(product, plant, period)]
                ), f"Inventory_Balance_{product}_{plant}_P{period}"
            else:
                nu_model += (
                    inventory[(product, plant, period - 1)]
                    + production[(product, plant, period)]
                    == shipment[(product, plant, period)]
                    + inventory[(product, plant, period)]
                ), f"Inventory_Balance_{product}_{plant}_P{period}"

# demand constraints
# Period 1 only includes the required contract demand and then Periods 2 through 5 include contract demand plus demand created by prior advertising.
for product in products:
    for period in periods:
        if period == 1:
            required_demand = base_demand[(product, period)]
        else:
            required_demand = (
                base_demand[(product, period)]
                + advertising[(product, period - 1)]
            )

        nu_model += (
            lpSum(
                shipment[(product, plant, period)]
                for plant in plants
            )
            == required_demand
        ), f"Demand_{product}_P{period}"

# labor balance constraints
# Production labor must be covered by regular labor and overtime labor
for plant in plants:
    for period in periods:
        nu_model += (
            lpSum(
                labor_hours[(plant, product)]
                * production[(product, plant, period)]
                for product in products
            )
            == regular_labor[(plant, period)]
            + overtime_labor[(plant, period)]
        ), f"Labor_Balance_{plant}_P{period}"

# inventory storage capacity constraints
for plant in plants:
    for period in periods[:-1]:
        nu_model += (
            lpSum(
                inventory[(product, plant, period)]
                for product in products
            )
            <= inventory_capacity[plant]
        ), f"Inventory_Capacity_{plant}_P{period}"

# raw material 1 constraints
# The vendor can provide a total of 70 tons across both plants during each period
for period in periods:
    nu_model += (
        lpSum(
            raw_material_1[(plant, product)]
            * production[(product, plant, period)]
            for product in products
            for plant in plants
        )
        <= raw_material_1_capacity
    ), f"Raw_Material_1_P{period}"

# raw material 2 constraints
# The vendor can provide a total of 2.5 tons across both plants during each period
for period in periods:
    nu_model += (
        lpSum(
            raw_material_2[(plant, product)]
            * production[(product, plant, period)]
            for product in products
            for plant in plants
        )
        <= raw_material_2_capacity
    ), f"Raw_Material_2_P{period}"

# total advertising budget constraint
nu_model += (
    lpSum(
        advertising_cost[product] * advertising[(product, period)]
        for product in products
        for period in advertising_periods
    )
    <= total_advertising_budget
), "Advertising_Budget"

# There is no inventory at the end of the planning horizon
for product in products:
    for plant in plants:
        nu_model += (
            inventory[(product, plant, 5)] == 0
        ), f"Ending_Inventory_{product}_{plant}"

## Solve the Baseline Model

In [8]:
# solve the problem
nu_model.writeLP("NU_Industries_Baseline.lp")
nu_model.solve(GLPK(msg=False))
print("Status:", LpStatus[nu_model.status])

print("Maximum Profit = $", format(round(value(nu_model.objective), 1), ","))
print("")


Status: Optimal
Maximum Profit = $ 7,066,035.3



## Baseline Results

In [9]:
# create a production, shipment, and inventory results table
results = []

for product in products:
    for plant in plants:
        for period in periods:
            results.append({
                "Product": product,
                "Plant": plant,
                "Period": period,
                "Production": production[(product, plant, period)].varValue,
                "Shipment": shipment[(product, plant, period)].varValue,
                "Ending Inventory": inventory[(product, plant, period)].varValue
            })

results_df = pd.DataFrame(results)

# display only rows that contain activity
active_results_df = results_df[
    (results_df["Production"] > 0.0001)
    | (results_df["Shipment"] > 0.0001)
    | (results_df["Ending Inventory"] > 0.0001)
].copy()

active_results_df.round(1)

,Product,Plant,Period,Production,Shipment,Ending Inventory
0,Widget,Plant_A,1,61.4,61.4,0.0
2,Widget,Plant_A,3,89.8,89.8,0.0
3,Widget,Plant_A,4,80.1,80.1,0.0
4,Widget,Plant_A,5,83.8,83.8,0.0
5,Widget,Plant_B,1,8.6,8.6,0.0
6,Widget,Plant_B,2,125.0,125.0,0.0
7,Widget,Plant_B,3,95.2,95.2,0.0
8,Widget,Plant_B,4,109.9,109.9,0.0
9,Widget,Plant_B,5,116.2,116.2,0.0
10,Gadget,Plant_A,1,270.0,200.0,70.0


## Additional Results

**Replace this Markdown with code create and report the advertising, labor, and detailed profit summary outputs.**

### Profit Summary

In [10]:
# create a detailed profit summary
profit_summary = pd.DataFrame({
    "Component": [
        "Sales Revenue",
        "Raw Material Cost",
        "Regular Labor Cost",
        "Overtime Labor Cost",
        "Inventory Holding Cost",
        "Transportation Cost",
        "Advertising Cost",
        "Maximum Profit"
    ],
    "Amount": [
        value(total_revenue),
        -value(total_raw_material_cost),
        -value(total_regular_labor_cost),
        -value(total_overtime_labor_cost),
        -value(total_inventory_cost),
        -value(total_shipping_cost),
        -value(total_advertising_cost),
        value(nu_model.objective)
    ]
})

profit_summary["Amount"] = profit_summary["Amount"].map(
    lambda x: f"${x:,.2f}"
)

profit_summary

,Component,Amount
0,Sales Revenue,"$8,480,606.99"
1,Raw Material Cost,"$-965,351.10"
2,Regular Labor Cost,"$-334,324.09"
3,Overtime Labor Cost,"$-29,221.69"
4,Inventory Holding Cost,"$-1,016.69"
5,Transportation Cost,"$-18,557.68"
6,Advertising Cost,"$-66,100.42"
7,Maximum Profit,"$7,066,035.31"


### Advertising Results

In [11]:
# create advertising summary
advertising_results = []

for p in products:
    for t in periods[:4]:
        advertising_results.append({
            "Product": p,
            "Period": t,
            "Additional Demand": advertising[(p, t)].varValue,
            "Advertising Cost": advertising[(p, t)].varValue * advertising_cost[p]
        })

advertising_results = pd.DataFrame(advertising_results)

advertising_results

,Product,Period,Additional Demand,Advertising Cost
0,Widget,1,0.000000,0.000000
1,Widget,2,0.000000,0.000000
2,Widget,3,0.000000,0.000000
3,Widget,4,0.000000,0.000000
4,Gadget,1,0.000000,0.000000
5,Gadget,2,0.000000,0.000000
6,Gadget,3,0.000000,0.000000
7,Gadget,4,0.000000,0.000000
8,Flugel,1,181.481481,32666.666667
9,Flugel,2,96.937522,17448.753982


### Labor Results

In [12]:
# create labor summary
labor_results = []

for plant in plants:
    for period in periods:
        labor_results.append({
            "Plant": plant,
            "Period": period,
            "Regular Labor": regular_labor[(plant, period)].varValue,
            "Overtime Labor": overtime_labor[(plant, period)].varValue,
            "Total Labor": (
                regular_labor[(plant, period)].varValue
                + overtime_labor[(plant, period)].varValue
            )
        })

labor_results = pd.DataFrame(labor_results)

labor_results.round(1)

,Plant,Period,Regular Labor,Overtime Labor,Total Labor
0,Plant_A,1,2500.0,0.0,2500.0
1,Plant_A,2,1725.6,0.0,1725.6
2,Plant_A,3,2500.0,0.0,2500.0
3,Plant_A,4,2500.0,0.0,2500.0
4,Plant_A,5,2500.0,0.0,2500.0
5,Plant_B,1,1952.5,0.0,1952.5
6,Plant_B,2,3800.0,1116.2,4916.2
7,Plant_B,3,3800.0,266.9,4066.9
8,Plant_B,4,3800.0,159.1,3959.1
9,Plant_B,5,3800.0,169.2,3969.2


### Raw Material Usage

In [13]:
# create raw material usage summary
raw_material_results = []

for period in periods:
    rm1_used = sum(
        raw_material_1[(plant, product)]
        * production[(product, plant, period)].varValue
        for product in products
        for plant in plants
    )

    rm2_used = sum(
        raw_material_2[(plant, product)]
        * production[(product, plant, period)].varValue
        for product in products
        for plant in plants
    )

    raw_material_results.append({
        "Period": period,
        "Raw Material 1 Used": rm1_used,
        "Raw Material 1 Capacity": raw_material_1_capacity,
        "Raw Material 2 Used": rm2_used,
        "Raw Material 2 Capacity": raw_material_2_capacity
    })

raw_material_results = pd.DataFrame(raw_material_results)

raw_material_results.round(1)

,Period,Raw Material 1 Used,Raw Material 1 Capacity,Raw Material 2 Used,Raw Material 2 Capacity
0,1,110678.2,140000,2119.2,5000.0
1,2,140000.0,140000,5000.0,5000.0
2,3,140000.0,140000,4909.0,5000.0
3,4,140000.0,140000,4714.4,5000.0
4,5,140000.0,140000,4756.4,5000.0


## Model Validation

**Replace this Markdown with code for formal model validation checks.**

The validation section should eventually confirm that demand is met, resource limits are followed, the advertising budget is not exceeded, and ending inventory matches the case requirements.

### Advertising Budget Check

In [14]:
# validate advertising budget
advertising_spending = value(total_advertising_cost)

advertising_validation = pd.DataFrame({
    "Budget Used": [advertising_spending],
    "Budget Limit": [total_advertising_budget],
    "Remaining Budget": [total_advertising_budget - advertising_spending],
    "Status": [
        "PASS" if advertising_spending <= total_advertising_budget else "FAIL"
    ]
})

advertising_validation.round(1)

,Budget Used,Budget Limit,Remaining Budget,Status
0,66100.4,70000,3899.6,PASS


### Raw Material Capacity Check

In [15]:
# validate raw material capacity
tolerance = 1e-6

raw_material_validation = raw_material_results.copy()

raw_material_validation["Raw Material 1 Status"] = raw_material_validation.apply(
    lambda row: "PASS"
    if row["Raw Material 1 Used"] <= row["Raw Material 1 Capacity"] + tolerance
    else "FAIL",
    axis=1
)

raw_material_validation["Raw Material 2 Status"] = raw_material_validation.apply(
    lambda row: "PASS"
    if row["Raw Material 2 Used"] <= row["Raw Material 2 Capacity"] + tolerance
    else "FAIL",
    axis=1
)

raw_material_validation.round(1)

,Period,Raw Material 1 Used,Raw Material 1 Capacity,Raw Material 2 Used,Raw Material 2 Capacity,Raw Material 1 Status,Raw Material 2 Status
0,1,110678.2,140000,2119.2,5000.0,PASS,PASS
1,2,140000.0,140000,5000.0,5000.0,PASS,PASS
2,3,140000.0,140000,4909.0,5000.0,PASS,PASS
3,4,140000.0,140000,4714.4,5000.0,PASS,PASS
4,5,140000.0,140000,4756.4,5000.0,PASS,PASS


In [16]:
# delete
raw_material_validation.loc[3, "Raw Material 1 Used"]

np.float64(140000.0000000007)

### Labor Capacity Check

In [17]:
# validate labor capacity
labor_validation = labor_results.copy()

labor_validation["Regular Labor Capacity"] = labor_validation.apply(
    lambda row: regular_labor_capacity[(row["Plant"], row["Period"])],
    axis=1
)

labor_validation["Status"] = labor_validation.apply(
    lambda row: "PASS"
    if row["Regular Labor"] <= row["Regular Labor Capacity"] + tolerance
    else "FAIL",
    axis=1
)

labor_validation.round(1)

,Plant,Period,Regular Labor,Overtime Labor,Total Labor,Regular Labor Capacity,Status
0,Plant_A,1,2500.0,0.0,2500.0,2500,PASS
1,Plant_A,2,1725.6,0.0,1725.6,2500,PASS
2,Plant_A,3,2500.0,0.0,2500.0,2500,PASS
3,Plant_A,4,2500.0,0.0,2500.0,2500,PASS
4,Plant_A,5,2500.0,0.0,2500.0,2500,PASS
5,Plant_B,1,1952.5,0.0,1952.5,3800,PASS
6,Plant_B,2,3800.0,1116.2,4916.2,3800,PASS
7,Plant_B,3,3800.0,266.9,4066.9,3800,PASS
8,Plant_B,4,3800.0,159.1,3959.1,3800,PASS
9,Plant_B,5,3800.0,169.2,3969.2,3800,PASS


### Ending Inventory Check

In [18]:
# validate ending inventory
ending_inventory_results = []

for product in products:
    for plant in plants:
        ending_inventory_results.append({
            "Product": product,
            "Plant": plant,
            "Ending Inventory": inventory[(product, plant, 5)].varValue,
            "Status": (
                "PASS"
                if abs(inventory[(product, plant, 5)].varValue) <= tolerance
                else "FAIL"
            )
        })

ending_inventory_validation = pd.DataFrame(ending_inventory_results)

ending_inventory_validation.round(1)

,Product,Plant,Ending Inventory,Status
0,Widget,Plant_A,0.0,PASS
1,Widget,Plant_B,0.0,PASS
2,Gadget,Plant_A,0.0,PASS
3,Gadget,Plant_B,0.0,PASS
4,Flugel,Plant_A,0.0,PASS
5,Flugel,Plant_B,0.0,PASS


### Inventory Capacity Check

In [19]:
# validate inventory capacity
inventory_capacity_results = []

for plant in plants:
    for period in periods[:4]:
        total_inventory = sum(
            inventory[(product, plant, period)].varValue
            for product in products
        )

        inventory_capacity_results.append({
            "Plant": plant,
            "Period": period,
            "Inventory Used": total_inventory,
            "Inventory Capacity": inventory_capacity[plant],
            "Status": (
                "PASS"
                if total_inventory <= inventory_capacity[plant] + tolerance
                else "FAIL"
            )
        })

inventory_capacity_validation = pd.DataFrame(inventory_capacity_results)

inventory_capacity_validation.round(1)

,Plant,Period,Inventory Used,Inventory Capacity,Status
0,Plant_A,1,70.0,70,PASS
1,Plant_A,2,63.0,70,PASS
2,Plant_A,3,0.0,70,PASS
3,Plant_A,4,0.0,70,PASS
4,Plant_B,1,50.0,50,PASS
5,Plant_B,2,0.0,50,PASS
6,Plant_B,3,0.0,50,PASS
7,Plant_B,4,0.0,50,PASS


### Demand Satisfaction Check

In [20]:
# validate demand satisfaction
demand_validation_results = []

for product in products:
    for period in periods:
        total_shipments = sum(
            shipment[(product, plant, period)].varValue
            for plant in plants
        )

        base_required_demand = base_demand[(product, period)]

        if period == 1:
            advertising_demand = 0
        else:
            advertising_demand = advertising[(product, period - 1)].varValue

        required_demand = base_required_demand + advertising_demand

        demand_validation_results.append({
            "Product": product,
            "Period": period,
            "Base Demand": base_required_demand,
            "Advertising Demand": advertising_demand,
            "Required Demand": required_demand,
            "Shipments": total_shipments,
            "Status": (
                "PASS"
                if abs(total_shipments - required_demand) <= tolerance
                else "FAIL"
            )
        })

demand_validation = pd.DataFrame(demand_validation_results)

demand_validation.round(1)

,Product,Period,Base Demand,Advertising Demand,Required Demand,Shipments,Status
0,Widget,1,70,0.0,70.0,70.0,PASS
1,Widget,2,125,0.0,125.0,125.0,PASS
2,Widget,3,185,0.0,185.0,185.0,PASS
3,Widget,4,190,0.0,190.0,190.0,PASS
4,Widget,5,200,0.0,200.0,200.0,PASS
5,Gadget,1,200,0.0,200.0,200.0,PASS
6,Gadget,2,300,0.0,300.0,300.0,PASS
7,Gadget,3,295,0.0,295.0,295.0,PASS
8,Gadget,4,245,0.0,245.0,245.0,PASS
9,Gadget,5,240,0.0,240.0,240.0,PASS


## Sensitivity Analysis and Integer Programming

**Replace this Markdown with code after these sections are completed.**

The group plans to review the baseline solution with the instructor before adding sensitivity analysis, advertising and raw-material scenarios, and the integer programming comparison.